# Project A - Session 0: compactor precondition

Establishes whether the compactor's keep/drop decision carries any salience signal.
**Nothing downstream is worth running until this passes.**

Pass condition: salience lift beats the positional control printed alongside it, with
zero fallbacks and zero empty keeps.

Expected: only `Qwen2.5-1.5B-Instruct` with `backend: scoring` clears it (2.56x vs a
1.68x control on 6 eval trajectories). Budget ~1 GPU-hour.

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/myrios'
if not os.path.exists(REPO):
    r = subprocess.run(['git', 'clone', 'https://github.com/USER/myrios.git', REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
    assert r.returncode == 0, 'clone failed - fix the URL in this cell'
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))


In [ ]:
run(f"python scripts/preflight.py --config configs/kaggle.yaml --require-gpu")


## Datasets

Three sets, because the lift figure differs sharply between them:

- **synthetic (marked)** — facts introduced by `One thing to lock in:`. Debug and
  ratio-sweep set. Measured 2.56x against a 1.68x control.
- **synthetic (unmarked)** — same generator, marker removed. Floor case: facts and
  filler share one template bank and one register. Measured 1.10x against 1.57x.
- **HotpotQA** — natural prose, no marker, supporting sentences scattered through an
  early window. **Primary set.** Measured 1.61x against a 0.98x control at n=60.

The CPU numbers above all used 6 eval trajectories. The point of this session is to
widen them: HotpotQA's margin is only about 2.6 sigma at that size.

In [ ]:
run(f"python data/generate_synthetic.py --n-train 48 --n-eval 24 --n-turns 120")
run(f"python data/generate_synthetic.py --n-train 48 --n-eval 24 --n-turns 120 --unmarked --out artifacts/data/synthetic_hard")
run(f"python data/load_hotpotqa.py --n-train 32 --n-eval 24 --per-trajectory 4")


## Scorer hand-check

Cheap sanity pass before the full sweep: do obviously salient lines outscore obvious
filler? A non-positive separation means the sweep is pointless.

In [ ]:
for m in ['HuggingFaceTB/SmolLM2-360M-Instruct', 'Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']:
    run(f"python compactor/inspect_scorer.py --config configs/kaggle.yaml --set model.base={m}")


## Salience lift per compactor

Both elicitation formats, so the index-list failure is reproduced on GPU rather than
taken on trust from the CPU runs.

In [ ]:
MODELS = ['HuggingFaceTB/SmolLM2-360M-Instruct', 'Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']
SETS = [
    ('synthetic', 'configs/kaggle.yaml', 'artifacts/data/synthetic'),
    ('unmarked', 'configs/kaggle.yaml', 'artifacts/data/synthetic_hard'),
    ('hotpotqa', 'configs/kaggle_hotpotqa.yaml', 'artifacts/data/hotpotqa'),
]
for name, cfg, ddir in SETS:
    for backend in ['scoring', 'model']:
        for m in MODELS:
            slug = f"{name}_{m.split('/')[-1]}_{backend}"
            out = f'/kaggle/working/artifacts/runs/compactor_check/{slug}'
            print('=' * 70, slug)
            run(f"python baselines/cascading.py --config {cfg} --split eval --out {out} --set model.base={m} compaction.backend={backend} data.dir={ddir}")
            run(f"python eval/span_report.py --events {out}/eval_events.jsonl --show 0 --out {out}/span_report.json")


In [ ]:
import json, glob
rows = []
for p in sorted(glob.glob('/kaggle/working/artifacts/runs/compactor_check/*/span_report.json')):
    r = json.load(open(p))
    rows.append((p.split('/')[-2], r['fact_keep_rate'], r['filler_keep_rate'],
                 r['salience_lift'], r['positional_control']['salience_lift']))
print(f"{'config':<40} {'fact':>6} {'filler':>7} {'lift':>7} {'control':>8}  verdict")
for name, f, fl, lift, ctrl in rows:
    verdict = 'USABLE' if lift > ctrl * 1.1 else 'no signal'
    print(f'{name:<40} {f:6.3f} {fl:7.3f} {lift:7.2f} {ctrl:8.2f}  {verdict}')

In [ ]:
run(f"python scripts/kaggle_sync.py save --run-root /kaggle/working/artifacts/runs --archive /kaggle/working/runs.zip")
print('Save /kaggle/working/runs.zip as a Kaggle Dataset before the session ends.')